# 20 Hz Real-Time Control Loop Demo
**Bonus track (2 pts)** — SmolVLA inference driving a LIBERO-Spatial simulator at >= 20 Hz.

**Prerequisites:** Upload `inference.py` and `demo_realtime_20hz.py` to `/content/` before running.

**Runtime:** Switch to a GPU runtime (A100 or T4). Total runtime ~15 min.

In [ ]:
import torch, os
assert torch.cuda.is_available(), "No GPU -- switch runtime"
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")
print(f"PyTorch {torch.__version__} | CUDA {torch.version.cuda}")

In [ ]:
# Install lerobot with LIBERO support + video recording (~5 min)
!pip install -q "lerobot[libero] @ git+https://github.com/huggingface/lerobot.git"
!pip install -q num2words "imageio[ffmpeg]"

# Fix egl_probe cmake version (known issue)
!git clone -q https://github.com/StanfordVL/egl_probe.git /tmp/egl_probe 2>/dev/null || true
!cd /tmp/egl_probe && sed -i 's/cmake_minimum_required(VERSION 2.8.12)/cmake_minimum_required(VERSION 3.10)/' egl_probe/CMakeLists.txt && pip install -q . 2>/dev/null

os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import mujoco
print(f"MuJoCo {mujoco.__version__} | MUJOCO_GL={os.environ['MUJOCO_GL']}")

In [ ]:
# Verify both files are present
assert os.path.exists('/content/inference.py'), 'Upload inference.py to /content/'
assert os.path.exists('/content/demo_realtime_20hz.py'), 'Upload demo_realtime_20hz.py to /content/'
print('Files found. Running demo...')

In [ ]:
!cd /content && python demo_realtime_20hz.py \
    --model-path lerobot/smolvla_base \
    --n-episodes 3 \
    --output demo_results.json

In [ ]:
# Display results
import json
with open('/content/demo_results.json') as f:
    r = json.load(f)
s = r['summary']
print(f"Policy Hz (amortized): {s['policy_amortized_hz']}")
print(f"Replan latency: {s['policy_replan_mean_ms']} ms")
print(f"Cached step: {s['policy_cached_mean_ms']} ms")
print(f"20 Hz met: {s['meets_20hz_policy']}")